# Modelica to PyPowSyBl: Dynamic Simulation Toolkit (Kundur Two-Area System, Model Compilation)

### Industrialized Workflow
1. **Create Network:** Programmatically build the Kundur two-area system via PyPowSyBl: 4 generators with their step-up transformers, 8 transmission lines with a weak tie between the areas, 2 loads and 2 shunt capacitors, in two voltage levels (20 kV and 230 kV).
2. **Initialize Loadflow:** Define load and generation profiles, with a transfer of 400 MW from area 1 to area 2.
3. **Run Loadflow:** Execute the base AC Loadflow using OpenLF.
4. **Compile Dynamic Models:** Compile the dynamic models with the Dynawo model generator instead of using the models precompiled in the Dynawo database: the standard generator models without and with power system stabilizer (PSS), the load model, and a variant of our own that adds a speed governor.
5. **Dynamic Models Setup:** Map the generators and the loads to the compiled models.
6. **Physical & Initialization Parameters:** Build the `.PAR` files using physical parameters and Loadflow initialization references.
7. **Execute Dynamic Simulations:** Simulate the same event with the precompiled and the compiled standard models, and with the compiled variant, and compare them.
8. **Analyse the Inter-Area Oscillation:** Estimate the frequency and the damping of the oscillation between the two areas, and the frequency reached after the event.

## Step 0: Environment Setup (Dynawo Install & Config)

Before running dynamic simulations, we need the Dynawo engine installed and configured so PyPowSyBl can find it.

In [ ]:
# --- 1. ENVIRONMENT SETUP: CHECK DYNAWO INSTALLATION ---
import os
import re
import datetime


def verify_dynawo_installation():
    """Finds and validates the Dynawo installation path from the system configuration."""
    config_directory = os.path.expanduser("~/.itools")
    config_filepath = os.path.join(config_directory, "config.yml")
    dynawo_path = None

    # Check if config.yml exists and extract the homeDir value using regex
    if os.path.exists(config_filepath):
        with open(config_filepath, "r") as file:
            content = file.read()
            match = re.search(r'homeDir:\s*["\']?([^"\'\n]+)["\']?', content)
            if match:
                dynawo_path = match.group(1)

    # Verify if a path was extracted and if it actually exists on the machine
    if not dynawo_path or not os.path.exists(dynawo_path):
        raise FileNotFoundError(
            f"Error! No valid Dynawo installation found on the system. "
            f"Detected path: '{dynawo_path}'. Make sure Dynawo is installed and properly configured in {config_filepath}."
        )

    print(f"Configuration correct! Dynawo installation validated at: {dynawo_path}")

    # Save the verified path in an environment variable for subsequent steps
    os.environ["DYNAWO_SYS_PATH"] = dynawo_path
    return dynawo_path, config_directory


# Execute verification
dynawo_sys_path, global_config_dir = verify_dynawo_installation()

In [ ]:
# --- 2. ENVIRONMENT SETUP: CONFIGURE TEMPLATES ---
def configure_environment_template(dynawo_path, config_directory):
    """Overwrites the PyPowSyBl configuration with the local project template."""
    target_config_filepath = os.path.join(config_directory, "config.yml")
    template_config_filepath = "./getting_started_data/config.yml"
    current_working_directory = os.getcwd()

    if not dynawo_path:
        raise ValueError("The Dynawo path is not defined. Please run the previous cell first.")

    # Create a backup of the existing configuration if it exists
    if os.path.exists(target_config_filepath):
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_filepath = os.path.join(config_directory, f"config_{timestamp}.yml")
        os.rename(target_config_filepath, backup_filepath)
        print(f"Backup created at: {backup_filepath}")

    with open(template_config_filepath, "r") as file:
        template_content = file.read()

    # Inject dynamic paths into the template
    template_content = template_content.replace('"WORKING_DIR/dynawo"', f'"{dynawo_path}"')
    template_content = template_content.replace("WORKING_DIR", current_working_directory)

    os.makedirs(config_directory, exist_ok=True)
    with open(target_config_filepath, "w") as file:
        file.write(template_content)

    print(f"\nDone! Template successfully configured and saved to {target_config_filepath}:\n")
    print(template_content)


# Execute configuration
configure_environment_template(dynawo_sys_path, global_config_dir)

In [ ]:
# --- 3. IMPORT LIBRARIES AND BASE CONFIGURATION ---
import contextlib
import json
import shutil
import subprocess
import time
import xml.etree.ElementTree as ET
from itertools import pairwise
from pathlib import Path

import numpy as np
import pandas as pd
import pypowsybl as pp
import pypowsybl.dynamic as dyn
import matplotlib.pyplot as plt
from IPython.display import display

# Compilation toolchain of the toolkit: checks of the Dynawo distribution and of the compiler
from python_pypowsybl.Code.core.model_compiler import DynawoToolchain

# Ensure plots are displayed inline within the Jupyter environment
%matplotlib inline

print(f"PyPowSyBl version: {pp.__version__}")

## Step 1 & 2: Create Network and Initialize Loadflow Profiles

We construct the two-area system of P. Kundur, *Power System Stability and Control* (Example 12.6), a reference case for the study of inter-area oscillations. Two identical areas, each with two 900 MVA generators, are connected by a weak tie of two circuits of 220 km:

| Element | Count | Data |
|---|---|---|
| Generators `GEN_1` … `GEN_4` | 4 | 900 MVA, 20 kV: `GEN_1` and `GEN_2` in area 1, `GEN_3` and `GEN_4` in area 2 |
| Step-up transformers `TR_1` … `TR_4` | 4 | 20/230 kV, 900 MVA, $X = 0.15$ pu |
| Lines | 8 | 230 kV, $r = 0.0001$, $x = 0.001$ and $b = 0.00175$ pu/km (100 MVA base): 25 km and 10 km in each area, and two circuits of 110 km from `BUS_7` to `BUS_8` and from `BUS_8` to `BUS_9` |
| Loads `LOAD_7`, `LOAD_9` | 2 | 967 MW + 100 Mvar in area 1, 1767 MW + 100 Mvar in area 2 |
| Shunt capacitors `SHUNT_7`, `SHUNT_9` | 2 | 200 Mvar and 350 Mvar |

The buses keep the numbering of the book: the generators are connected to `BUS_1` … `BUS_4` (20 kV), and the transmission network uses `BUS_5` … `BUS_11` (230 kV). Each generator substation holds the two voltage levels of its step-up transformer. The per unit data of the book are converted to ohms and siemens, the units of IIDM (on the 230 kV side for the transformers).

In [ ]:
# --- 4. CREATE NETWORK AND INITIALIZE PROFILES ---
# Base of the per unit data of the book: 100 MVA, and 230 kV in the transmission network
BASE_POWER = 100.0  # MVA
TRANSMISSION_VOLTAGE = 230.0  # kV
GENERATION_VOLTAGE = 20.0  # kV
IMPEDANCE_BASE = TRANSMISSION_VOLTAGE**2 / BASE_POWER  # Ohm


def build_kundur_two_area_network() -> pp.network.Network:
    """
    Programmatically constructs the Kundur two-area system: 4 generators with their step-up
    transformers, 8 transmission lines, 2 loads and 2 shunt capacitors.
    """
    power_network = pp.network.load("getting_started_data/Base_Case.xiidm")

    # Create Core Topology (7 Substations, 11 Voltage Levels and Buses). A generator
    # substation holds the generator bus (20 kV) and the transmission bus (230 kV) of its
    # step-up transformer, a load substation a transmission bus
    substation_buses = {
        "SUB_1": [1, 5],
        "SUB_2": [2, 6],
        "SUB_3": [3, 11],
        "SUB_4": [4, 10],
        "SUB_7": [7],
        "SUB_8": [8],
        "SUB_9": [9],
    }
    bus_numbers = [bus for buses in substation_buses.values() for bus in buses]
    power_network.create_substations(
        id=list(substation_buses),
        name=[f"Substation {substation[4:]}" for substation in substation_buses],
    )
    power_network.create_voltage_levels(
        id=[f"VL_{bus}" for bus in bus_numbers],
        name=[f"VL_{bus}" for bus in bus_numbers],
        substation_id=[substation for substation, buses in substation_buses.items() for _ in buses],
        topology_kind=["BUS_BREAKER"] * len(bus_numbers),
        nominal_v=[GENERATION_VOLTAGE if bus <= 4 else TRANSMISSION_VOLTAGE for bus in bus_numbers],
    )
    power_network.create_buses(
        id=[f"BUS_{bus}" for bus in bus_numbers],
        name=[f"BUS_{bus}" for bus in bus_numbers],
        voltage_level_id=[f"VL_{bus}" for bus in bus_numbers],
    )

    # Add 4 Generators: active power and voltage set point (pu) of the book
    generator_buses = [1, 2, 3, 4]
    gen_ids = [f"GEN_{i}" for i in generator_buses]
    power_network.create_generators(
        id=gen_ids,
        name=gen_ids,
        voltage_level_id=[f"VL_{i}" for i in generator_buses],
        bus_id=[f"BUS_{i}" for i in generator_buses],
        target_p=[700.0, 700.0, 719.0, 700.0],
        target_v=[v * GENERATION_VOLTAGE for v in (1.03, 1.01, 1.03, 1.01)],
        min_p=[0.0] * 4,
        max_p=[900.0] * 4,
        rated_s=[900.0] * 4,
        voltage_regulator_on=[True] * 4,
    )

    # Add 4 Step-up Transformers 20/230 kV: X = 0.15 pu on 900 MVA, in ohms at the 230 kV side
    transmission_buses = [5, 6, 11, 10]
    transformer_ids = [f"TR_{i}" for i in generator_buses]
    power_network.create_2_windings_transformers(
        id=transformer_ids,
        name=transformer_ids,
        voltage_level1_id=[f"VL_{i}" for i in generator_buses],
        bus1_id=[f"BUS_{i}" for i in generator_buses],
        voltage_level2_id=[f"VL_{i}" for i in transmission_buses],
        bus2_id=[f"BUS_{i}" for i in transmission_buses],
        rated_u1=[GENERATION_VOLTAGE] * 4,
        rated_u2=[TRANSMISSION_VOLTAGE] * 4,
        rated_s=[900.0] * 4,
        r=[0.0] * 4,
        x=[0.15 * TRANSMISSION_VOLTAGE**2 / 900.0] * 4,
        g=[0.0] * 4,
        b=[0.0] * 4,
    )

    # Add 2 Loads, one in each area
    power_network.create_loads(
        id=["LOAD_7", "LOAD_9"],
        name=["LOAD_7", "LOAD_9"],
        voltage_level_id=["VL_7", "VL_9"],
        bus_id=["BUS_7", "BUS_9"],
        p0=[967.0, 1767.0],
        q0=[100.0, 100.0],
    )

    # Add 2 Linear Shunt Capacitors (200 Mvar and 350 Mvar at 230 kV) for reactive support
    shunt_df = pd.DataFrame(
        {
            "id": ["SHUNT_7", "SHUNT_9"],
            "voltage_level_id": ["VL_7", "VL_9"],
            "bus_id": ["BUS_7", "BUS_9"],
            "model_type": ["LINEAR", "LINEAR"],
            "section_count": [1, 1],
        }
    ).set_index("id")

    linear_model_df = pd.DataFrame(
        {
            "id": ["SHUNT_7", "SHUNT_9"],
            "g_per_section": [0.0, 0.0],
            "b_per_section": [200.0 / TRANSMISSION_VOLTAGE**2, 350.0 / TRANSMISSION_VOLTAGE**2],
            "max_section_count": [1, 1],
        }
    ).set_index("id")

    power_network.create_shunt_compensators(shunt_df, linear_model_df)

    # Connect the buses with 8 Lines: {id: (bus 1, bus 2, length in km)}. The tie between the
    # areas is made of two circuits from BUS_7 to BUS_8, and two from BUS_8 to BUS_9
    line_data = {
        "LINE_5_6": (5, 6, 25.0),
        "LINE_6_7": (6, 7, 10.0),
        "LINE_7_8_1": (7, 8, 110.0),
        "LINE_7_8_2": (7, 8, 110.0),
        "LINE_8_9_1": (8, 9, 110.0),
        "LINE_8_9_2": (8, 9, 110.0),
        "LINE_9_10": (9, 10, 10.0),
        "LINE_10_11": (10, 11, 25.0),
    }
    lengths = [length for _, _, length in line_data.values()]

    power_network.create_lines(
        id=list(line_data),
        name=list(line_data),
        voltage_level1_id=[f"VL_{bus1}" for bus1, _, _ in line_data.values()],
        bus1_id=[f"BUS_{bus1}" for bus1, _, _ in line_data.values()],
        voltage_level2_id=[f"VL_{bus2}" for _, bus2, _ in line_data.values()],
        bus2_id=[f"BUS_{bus2}" for _, bus2, _ in line_data.values()],
        r=[0.0001 * length * IMPEDANCE_BASE for length in lengths],
        x=[0.001 * length * IMPEDANCE_BASE for length in lengths],
        # Half of the line charging at each end
        b1=[0.00175 * length / IMPEDANCE_BASE / 2 for length in lengths],
        b2=[0.00175 * length / IMPEDANCE_BASE / 2 for length in lengths],
    )

    return power_network


print("Creating the Kundur two-area network...")
power_network = build_kundur_two_area_network()
display(
    pd.Series(
        {
            "Substations": len(power_network.get_substations()),
            "Voltage levels": len(power_network.get_voltage_levels()),
            "Generators": len(power_network.get_generators()),
            "Transformers": len(power_network.get_2_windings_transformers()),
            "Lines": len(power_network.get_lines()),
            "Loads": len(power_network.get_loads()),
            "Shunt compensators": len(power_network.get_shunt_compensators()),
        },
        name="Count",
    ).to_frame()
)

## Step 3: Run Loadflow

Execute the base Loadflow using PyPowSyBl's OpenLF. The engine calculates the steady-state of the two areas: the voltage profile of the 11 buses (also in per unit, since the network has two voltage levels), the reactive power of the generators, and the power transferred from area 1 to area 2 through the tie (about 400 MW in the book).

In [ ]:
# --- 5. RUN STEADY-STATE LOADFLOW ---
def execute_and_display_loadflow(network: pp.network.Network):
    """Executes AC Load Flow analysis and displays the bus, generator and tie metrics."""
    print("Running Loadflow via OpenLF...")
    loadflow_parameters = pp.loadflow.Parameters()
    loadflow_results = pp.loadflow.run_ac(network, parameters=loadflow_parameters)

    print(f"Loadflow Status: {loadflow_results[0].status.name}")

    buses_dataframe = network.get_buses()
    nominal_voltages = network.get_voltage_levels()["nominal_v"]
    buses_dataframe["v_pu"] = (
        buses_dataframe["v_mag"] / nominal_voltages[buses_dataframe["voltage_level_id"]].values
    )
    display(buses_dataframe[["name", "v_mag", "v_pu", "v_angle"]])

    # PyPowSyBl gives the power flows at the terminals with the load sign convention
    generators_dataframe = network.get_generators()
    display(
        pd.DataFrame(
            {"P (MW)": -generators_dataframe["p"], "Q (Mvar)": -generators_dataframe["q"]}
        ).round(1)
    )

    tie_flow = network.get_lines().loc[["LINE_7_8_1", "LINE_7_8_2"], "p1"].sum()
    print(f"Power transferred from area 1 to area 2: {tie_flow:.1f} MW")


# Execute loadflow
execute_and_display_loadflow(power_network)

## Step 4: Compile the Dynamic Models

Instead of using the dynamic models precompiled in the Dynawo model database (`ddb`), this notebook compiles them from their Modelica definition with the Dynawo model generator (`dynawo.sh jobs --generate-preassembled`), as described in the [compilation guide](../../../Documentation/compilation_guide.md). Four models are compiled:

- `GeneratorSynchronousFourWindingsVRKundur`, `GeneratorSynchronousFourWindingsVRKundurPssKundur` and `LoadAlphaBeta`: the standard models of the Getting Started IV notebook, compiled from their official definition. They replace the precompiled models in a model database of this notebook, and must give the same results.
- `GeneratorSynchronousFourWindingsGoverPropVRKundurPssKundur`: a variant of our own, with the machine, exciter and stabilizer of `GeneratorSynchronousFourWindingsVRKundurPssKundur`, and the proportional speed governor of the Dynawo library (`GoverProportional`) instead of a constant mechanical power.

Each model is described by an architecture file, compiled into a library (`.so`), and its description (`.desc.xml`, the parameters and variables of the model) is dumped from the library. Compiling requires a full Dynawo distribution (headers and on-the-fly model compilation), built with a compiler compatible with the C++ compiler of the system (see the guide). Each model takes about half a minute to compile; the compiled libraries are reused by the next runs.

In [ ]:
# --- 6. COMPILATION SETUP: TOOLCHAIN AND DIRECTORIES ---
# The toolchain checks that the Dynawo distribution can compile models with the C++
# compiler of the system, and prepares the environment of the Dynawo model generator
# (compiler with the C++ ABI of the Dynawo libraries, Boost of the distribution)
toolchain = DynawoToolchain.detect(dynawo_sys_path)
print(f"Dynawo version: {toolchain.dynawo_version()}")
print(f"Toolchain: {toolchain.description}")

# Every file of the compilation goes to the export directory of the notebook
COMPILATION_DIRECTORY = Path("grid_export_GettingStarted_IV/compiled_models").resolve()
for sub_directory in ("model_lists", "build", "logs"):
    (COMPILATION_DIRECTORY / sub_directory).mkdir(parents=True, exist_ok=True)
compilation_environment = toolchain.environment(COMPILATION_DIRECTORY / "toolchain")
print(f"Compilation directory: {COMPILATION_DIRECTORY}")

In [ ]:
# --- 7. WRITE THE ARCHITECTURE FILES ---
# An architecture file (model list) describes the model to compile: a preassembled model
# made of the machine and its regulators, with the connections between them (connect) and
# between their initialization models (initConnect), or a single model such as the load.
# The id of the modelicaModel is the name of the compiled library, and the id of each unit
# dynamic model is the prefix of its parameters and variables (generator_, pss_...).
GENERATOR_MODEL = "GeneratorSynchronousFourWindingsVRKundur"
GENERATOR_PSS_MODEL = "GeneratorSynchronousFourWindingsVRKundurPssKundur"
LOAD_MODEL = "LoadAlphaBeta"
VARIANT_MODEL = "GeneratorSynchronousFourWindingsGoverPropVRKundurPssKundur"

ARCHITECTURES = {
    # Definitions of the Dynawo 1.7.0 model database
    GENERATOR_MODEL: """<?xml version='1.0' encoding='UTF-8'?>
<dyn:dynamicModelsArchitecture xmlns:dyn="http://www.rte-france.com/dynawo">
  <dyn:modelicaModel id="GeneratorSynchronousFourWindingsVRKundur">
    <dyn:unitDynamicModel id="generator" name="Dynawo.Electrical.Machines.OmegaRef.GeneratorSynchronous" initName="Dynawo.Electrical.Machines.OmegaRef.GeneratorSynchronousExt4W_INIT"/>
    <dyn:unitDynamicModel id="voltageRegulator" name="Dynawo.Electrical.Controls.Machines.VoltageRegulators.Standard.VRKundur" initName="Dynawo.Electrical.Controls.Machines.VoltageRegulators.Exciter_INIT"/>
    <dyn:initConnect id1="voltageRegulator" var1="Efd0Pu" id2="generator" var2="Efd0Pu"/>
    <dyn:initConnect id1="voltageRegulator" var1="Us0Pu" id2="generator" var2="UStator0Pu"/>
    <dyn:connect id1="voltageRegulator" var1="efdPu" id2="generator" var2="efdPu.value"/>
    <dyn:connect id1="voltageRegulator" var1="UsPu" id2="generator" var2="UStatorPu.value"/>
  </dyn:modelicaModel>
</dyn:dynamicModelsArchitecture>
""",
    GENERATOR_PSS_MODEL: """<?xml version='1.0' encoding='UTF-8'?>
<dyn:dynamicModelsArchitecture xmlns:dyn="http://www.rte-france.com/dynawo">
  <dyn:modelicaModel id="GeneratorSynchronousFourWindingsVRKundurPssKundur">
    <dyn:unitDynamicModel id="generator" name="Dynawo.Electrical.Machines.OmegaRef.GeneratorSynchronous" initName="Dynawo.Electrical.Machines.OmegaRef.GeneratorSynchronousExt4W_INIT"/>
    <dyn:unitDynamicModel id="voltageRegulator" name="Dynawo.Electrical.Controls.Machines.VoltageRegulators.Standard.VRKundur" initName="Dynawo.Electrical.Controls.Machines.VoltageRegulators.Exciter_INIT"/>
    <dyn:unitDynamicModel id="pss" name="Dynawo.Electrical.Controls.Machines.PowerSystemStabilizers.Standard.PssKundur"/>
    <dyn:initConnect id1="voltageRegulator" var1="Efd0Pu" id2="generator" var2="Efd0Pu"/>
    <dyn:initConnect id1="voltageRegulator" var1="Us0Pu" id2="generator" var2="UStator0Pu"/>
    <dyn:connect id1="voltageRegulator" var1="efdPu" id2="generator" var2="efdPu.value"/>
    <dyn:connect id1="voltageRegulator" var1="UsPu" id2="generator" var2="UStatorPu.value"/>
    <dyn:connect id1="voltageRegulator" var1="UPssPu" id2="pss" var2="UPssPu"/>
    <dyn:connect id1="pss" var1="omegaPu" id2="generator" var2="omegaPu.value"/>
  </dyn:modelicaModel>
</dyn:dynamicModelsArchitecture>
""",
    LOAD_MODEL: """<?xml version='1.0' encoding='UTF-8'?>
<dyn:dynamicModelsArchitecture xmlns:dyn="http://www.rte-france.com/dynawo">
  <dyn:modelicaModel id="LoadAlphaBeta">
    <dyn:unitDynamicModel id="load" name="Dynawo.Electrical.Loads.LoadAlphaBeta" initName="Dynawo.Electrical.Loads.Load_INIT"/>
  </dyn:modelicaModel>
</dyn:dynamicModelsArchitecture>
""",
    # Variant: the machine, exciter and stabilizer of the standard model with PSS, with the
    # proportional speed governor instead of a constant mechanical power (the connections of
    # the governor are those of GeneratorSynchronousFourWindingsProportionalRegulations)
    VARIANT_MODEL: """<?xml version='1.0' encoding='UTF-8'?>
<dyn:dynamicModelsArchitecture xmlns:dyn="http://www.rte-france.com/dynawo">
  <dyn:modelicaModel id="GeneratorSynchronousFourWindingsGoverPropVRKundurPssKundur">
    <dyn:unitDynamicModel id="generator" name="Dynawo.Electrical.Machines.OmegaRef.GeneratorSynchronous" initName="Dynawo.Electrical.Machines.OmegaRef.GeneratorSynchronousExt4W_INIT"/>
    <dyn:unitDynamicModel id="voltageRegulator" name="Dynawo.Electrical.Controls.Machines.VoltageRegulators.Standard.VRKundur" initName="Dynawo.Electrical.Controls.Machines.VoltageRegulators.Exciter_INIT"/>
    <dyn:unitDynamicModel id="pss" name="Dynawo.Electrical.Controls.Machines.PowerSystemStabilizers.Standard.PssKundur"/>
    <dyn:unitDynamicModel id="governor" name="Dynawo.Electrical.Controls.Machines.Governors.Simplified.GoverProportional" initName="Dynawo.Electrical.Controls.Machines.Governors.Governor_INIT"/>
    <dyn:initConnect id1="voltageRegulator" var1="Efd0Pu" id2="generator" var2="Efd0Pu"/>
    <dyn:initConnect id1="voltageRegulator" var1="Us0Pu" id2="generator" var2="UStator0Pu"/>
    <dyn:initConnect id1="governor" var1="Pm0Pu" id2="generator" var2="Pm0Pu"/>
    <dyn:connect id1="voltageRegulator" var1="efdPu" id2="generator" var2="efdPu.value"/>
    <dyn:connect id1="voltageRegulator" var1="UsPu" id2="generator" var2="UStatorPu.value"/>
    <dyn:connect id1="voltageRegulator" var1="UPssPu" id2="pss" var2="UPssPu"/>
    <dyn:connect id1="pss" var1="omegaPu" id2="generator" var2="omegaPu.value"/>
    <dyn:connect id1="governor" var1="omegaPu" id2="generator" var2="omegaPu.value"/>
    <dyn:connect id1="governor" var1="PmPu" id2="generator" var2="PmPu.value"/>
  </dyn:modelicaModel>
</dyn:dynamicModelsArchitecture>
""",
}

for model_name, architecture in ARCHITECTURES.items():
    (COMPILATION_DIRECTORY / "model_lists" / f"{model_name}.xml").write_text(architecture)
print(ARCHITECTURES[VARIANT_MODEL])

In [ ]:
# --- 8. COMPILE THE MODELS ---
def compile_model(model_name):
    """
    Compiles a model with the Dynawo model generator, then dumps its description, which
    also checks that Dynawo can load the library. A library compiled from the same
    architecture file is reused.
    """
    model_list = COMPILATION_DIRECTORY / "model_lists" / f"{model_name}.xml"
    library = COMPILATION_DIRECTORY / f"{model_name}.so"
    description = COMPILATION_DIRECTORY / f"{model_name}.desc.xml"
    compiled_list = COMPILATION_DIRECTORY / "build" / f"{model_name}.xml"
    if (
        description.exists()
        and compiled_list.exists()
        and compiled_list.read_text() == model_list.read_text()
    ):
        print(f"{model_name}: up to date, reused.")
        return

    launcher = str(toolchain.launcher)
    build_directory = COMPILATION_DIRECTORY / "build" / model_name
    shutil.rmtree(build_directory, ignore_errors=True)
    build_directory.mkdir(parents=True)
    start = time.time()

    # 1. Preassembly: Modelica model -> C++ code -> library and its external variables
    command = [launcher, "jobs", "--generate-preassembled", "--model-list", str(model_list)]
    command += ["--output-dir", str(build_directory)]
    print("$ " + " ".join(command))
    # The return code is checked after the output is saved to the log
    process = subprocess.run(
        command, check=False, capture_output=True, text=True, env=compilation_environment
    )
    log_file = COMPILATION_DIRECTORY / "logs" / f"{model_name}.log"
    log_file.write_text(process.stdout + process.stderr)
    if process.returncode != 0:
        raise RuntimeError(f"Compilation of {model_name} failed, see {log_file}")
    for suffix in (".so", ".extvar"):
        shutil.move(
            build_directory / f"{model_name}{suffix}",
            COMPILATION_DIRECTORY / f"{model_name}{suffix}",
        )

    # 2. Description of the model: its parameters and variables
    command = [launcher, "jobs", "--dump-model", "--model-file", str(library)]
    command += ["--output-file", str(description)]
    print("$ " + " ".join(command))
    subprocess.run(command, capture_output=True, check=True, env=compilation_environment)

    shutil.copyfile(model_list, compiled_list)
    print(f"{model_name}: compiled in {time.time() - start:.0f} s.")


for model_name in ARCHITECTURES:
    compile_model(model_name)

print("\nCompiled models:")
for model_name in ARCHITECTURES:
    for path in sorted(COMPILATION_DIRECTORY.glob(f"{model_name}.*")):
        print(f"- {path.name}")

In [ ]:
# --- 9. INSPECT THE COMPILED VARIANT ---
def read_model_parameters(model_name):
    """Reads the writable parameters of a compiled model from its description (desc.xml)."""
    root = ET.parse(COMPILATION_DIRECTORY / f"{model_name}.desc.xml").getroot()
    return pd.DataFrame(
        [
            {
                "name": parameter.get("name"),
                "type": parameter.get("valueType"),
                "default": parameter.get("defaultValue"),
            }
            for parameter in root.iter("{http://www.rte-france.com/dynawo}parameter")
            if parameter.get("readOnly") == "false"
        ]
    )


# Parameters of the new speed governor: those without default value must be given in
# the parameter file (Step 6)
variant_parameters = read_model_parameters(VARIANT_MODEL)
display(variant_parameters[variant_parameters["name"].str.startswith("governor_")])

In [ ]:
# --- 10. MODEL DATABASE OF THE SIMULATIONS ---
# PyPowSyBl cannot add model directories to the Dynawo jobs file it writes, and Dynawo
# refuses two libraries with the same name. The compiled models are therefore given to
# Dynawo through a model database of their own (DYNAWO_DDB_DIR): symbolic links to the
# model database of the installation, where the compiled models replace the standard
# models of the same name.
MODEL_DATABASE = COMPILATION_DIRECTORY / "model_database"
shutil.rmtree(MODEL_DATABASE, ignore_errors=True)
MODEL_DATABASE.mkdir()
compiled_files = {
    path.name: path
    for model_name in ARCHITECTURES
    for path in COMPILATION_DIRECTORY.glob(f"{model_name}.*")
}
for entry in (Path(dynawo_sys_path) / "ddb").iterdir():
    if entry.name not in compiled_files:
        (MODEL_DATABASE / entry.name).symlink_to(entry)
for name, path in compiled_files.items():
    (MODEL_DATABASE / name).symlink_to(path)
print(f"Model database: {MODEL_DATABASE} ({len(list(MODEL_DATABASE.iterdir()))} entries)")

# The variant is a new model for PyPowSyBl: it is registered as an additional model of the
# synchronous generator category
ADDITIONAL_MODELS_FILE = COMPILATION_DIRECTORY / "additional_models.json"
additional_models = {
    "SYNCHRONOUS_GENERATOR": {
        "libs": [
            {
                "lib": VARIANT_MODEL,
                "doc": "Four-winding synchronous generator with voltage regulator VRKundur, "
                "power system stabilizer PssKundur and governor GoverProportional, "
                "compiled by this notebook.",
            }
        ]
    }
}
ADDITIONAL_MODELS_FILE.write_text(json.dumps(additional_models, indent=4))


@contextlib.contextmanager
def dynawo_model_database(model_database):
    """Makes the Dynawo runs of the block read their models from a model database."""
    previous = os.environ.get("DYNAWO_DDB_DIR")
    if model_database is not None:
        os.environ["DYNAWO_DDB_DIR"] = str(model_database)
    try:
        yield
    finally:
        if previous is None:
            os.environ.pop("DYNAWO_DDB_DIR", None)
        else:
            os.environ["DYNAWO_DDB_DIR"] = previous

## Step 5: Dynamic Models Setup

Map the static elements to their dynamic models in three scenarios: without PSS and with PSS (the standard models of the Getting Started IV notebook), and with PSS and governor (the variant). The loads are mapped to `LoadAlphaBeta` in every scenario. The precompiled and the compiled standard models share their names: the model database given to Dynawo decides which libraries are loaded. The variant has its own name and its own parameter sets.

In [ ]:
# --- 11. CONFIGURE DYNAMIC MODELS MAPPING ---
GENERATORS = ["GEN_1", "GEN_2", "GEN_3", "GEN_4"]
LOADS = ["LOAD_7", "LOAD_9"]

# Generator model and suffix of the parameter sets of each scenario
SCENARIOS = {
    "Without PSS": {"model_name": GENERATOR_MODEL, "parameter_suffix": ""},
    "With PSS": {"model_name": GENERATOR_PSS_MODEL, "parameter_suffix": "_PSS"},
    "With PSS and governor": {"model_name": VARIANT_MODEL, "parameter_suffix": "_GOVERNOR"},
}


def map_dynamic_models(model_name, parameter_suffix="") -> dyn.ModelMapping:
    """Maps the generators to a dynamic model, and the loads to voltage-dependent loads."""
    dynamic_mapping = dyn.ModelMapping()

    generator_mapping_df = pd.DataFrame(
        [
            {
                "static_id": generator,
                "parameter_set_id": f"{generator}{parameter_suffix}",
                "model_name": model_name,
            }
            for generator in GENERATORS
        ]
    ).set_index("static_id", drop=False)
    dynamic_mapping.add_synchronous_generator(generator_mapping_df)

    load_mapping_df = pd.DataFrame(
        [{"static_id": load, "parameter_set_id": load, "model_name": LOAD_MODEL} for load in LOADS]
    ).set_index("static_id", drop=False)
    dynamic_mapping.add_base_load(load_mapping_df)
    return dynamic_mapping


scenario_mappings = {
    scenario: map_dynamic_models(**configuration) for scenario, configuration in SCENARIOS.items()
}
print(
    f"Dynamic mapping complete for {len(GENERATORS)} generators and {len(LOADS)} loads "
    f"in {len(SCENARIOS)} scenarios."
)

## Step 5: Physical Parameters and Loadflow Initialization

Construct the `.PAR` files from the data of the book:

- **Machines:** the generator data of Example 12.6 on the rating of the machines (900 MVA, 20 kV), with an inertia constant $H = 6.5$ s in area 1 and $H = 6.175$ s in area 2. The magnetic saturation is neglected, and the step-up transformers are modelled in the network, so the transformer of the machine model is left out.
- **Excitation system:** the thyristor exciter of the book ($K_A = 200$, $T_R = 0.01$ s), with the field voltage limits and the stabilizer of the Dynawo example `Kundur_Example13` (Kundur's Example 13.2).
- **Speed governor (variant):** `GoverProportional` modulates the mechanical power with the frequency deviation, $P_m = P_{m0} + K_{Gover} (\omega_{ref} - \omega)$ in pu of `PNom`, within the limits `PMin` and `PMax`: $K_{Gover} = 20$ is a droop of 5 %.
- **Loads:** the exponents $\alpha = 1$ and $\beta = 2$.

Each generator has a parameter set for each scenario, since the PSS parameters only exist in the model with PSS. IIDM references (`<reference>`) retrieve $P$, $Q$, $V$, and $\theta$ (in rad, `angle_pu`) from the Loadflow solution to guarantee seamless $t=0$ initialization.

In [ ]:
# --- 12. GENERATE PHYSICAL PARAMETERS (PAR FILES) ---
# Machine data of Example 12.6 on the rating of the machines (900 MVA, 20 kV)
MACHINE_PARAMETERS = {
    "generator_SNom": 900.0,
    # Nominal active powers (power factor of 0.9): the model requires PNomAlt != SNom
    "generator_PNomTurb": 810.0,
    "generator_PNomAlt": 810.0,
    "generator_UNom": 20.0,
    "generator_RaPu": 0.0025,
    "generator_XlPu": 0.2,
    "generator_XdPu": 1.8,
    "generator_XpdPu": 0.3,
    "generator_XppdPu": 0.25,
    "generator_Tpd0": 8.0,
    "generator_Tppd0": 0.03,
    "generator_XqPu": 1.7,
    "generator_XpqPu": 0.55,
    "generator_XppqPu": 0.25,
    "generator_Tpq0": 0.4,
    "generator_Tppq0": 0.05,
    "generator_DPu": 0.0,
    # No magnetic saturation
    "generator_md": 0.0,
    "generator_mq": 0.0,
    "generator_nd": 0.0,
    "generator_nq": 0.0,
    # The step-up transformers are in the network: no transformer in the machine model
    "generator_SnTfo": 900.0,
    "generator_UNomHV": 20.0,
    "generator_UNomLV": 20.0,
    "generator_UBaseHV": 20.0,
    "generator_UBaseLV": 20.0,
    "generator_RTfPu": 0.0,
    "generator_XTfPu": 0.0,
    # Base of the excitation voltage and approximations of the Dynawo example Kundur_Example13
    "generator_ExcitationPu": 1,
    "generator_MdPuEfd": 0.0,
    "generator_UseApproximation": True,
}

# Inertia constants (s): area 1 and area 2
INERTIA_CONSTANTS = {"GEN_1": 6.5, "GEN_2": 6.5, "GEN_3": 6.175, "GEN_4": 6.175}

# Thyristor exciter of the book, with the field voltage limits of Kundur_Example13
VOLTAGE_REGULATOR_PARAMETERS = {
    "voltageRegulator_Ka": 200.0,
    "voltageRegulator_tR": 0.01,
    "voltageRegulator_EfdMaxPu": 7.0,
    "voltageRegulator_EfdMinPu": -6.4,
}

# Power system stabilizer of Kundur_Example13
PSS_PARAMETERS = {
    "pss_KStab": 9.5,
    "pss_tW": 1.41,
    "pss_t1": 0.154,
    "pss_t2": 0.033,
    "pss_VsMinPu": -0.2,
    "pss_VsMaxPu": 0.2,
}

# Proportional speed governor of the variant (Step 4): a gain of 20 is a droop of 5 %
GOVERNOR_PARAMETERS = {
    "governor_KGover": 20.0,
    "governor_PNom": 810.0,
    "governor_PMin": 0.0,
    "governor_PMax": 810.0,
}

# Regulator parameters of each scenario, by suffix of the parameter sets (Step 4)
SCENARIO_PARAMETERS = {
    "": VOLTAGE_REGULATOR_PARAMETERS,
    "_PSS": {**VOLTAGE_REGULATOR_PARAMETERS, **PSS_PARAMETERS},
    "_GOVERNOR": {**VOLTAGE_REGULATOR_PARAMETERS, **PSS_PARAMETERS, **GOVERNOR_PARAMETERS},
}

# Voltage dependence of the loads: constant current P and constant impedance Q
LOAD_PARAMETERS = {"load_alpha": 1.0, "load_beta": 2.0}

# Initial values from the Loadflow solution: P, Q, V and the voltage angle in rad
INITIALIZATION_REFERENCES = {"P0Pu": "p_pu", "Q0Pu": "q_pu", "U0Pu": "v_pu", "UPhase0": "angle_pu"}


def parameter_line(name, value):
    """Renders a parameter, with the PAR type of its Python value."""
    if isinstance(value, bool):
        return f'        <par type="BOOL" name="{name}" value="{str(value).lower()}"/>\n'
    if isinstance(value, int):
        return f'        <par type="INT" name="{name}" value="{value}"/>\n'
    return f'        <par type="DOUBLE" name="{name}" value="{value}"/>\n'


def parameter_set(set_id, parameters, model_prefix):
    """Renders a parameter set, with the initialization references of the model."""
    xml_set = f'    <set id="{set_id}">\n'
    xml_set += "".join(parameter_line(name, value) for name, value in parameters.items())
    for name, iidm_name in INITIALIZATION_REFERENCES.items():
        xml_set += (
            f'        <reference type="DOUBLE" name="{model_prefix}_{name}" '
            f'origData="IIDM" origName="{iidm_name}"/>\n'
        )
    return xml_set + "    </set>\n"


def generate_parameter_files():
    """Writes the XML parameter files for the dynamic models and static network."""
    print("Generating dynamic physical parameters and Loadflow initialization mapping...")

    xml_dynamic_model_parameters = (
        '<?xml version="1.0" encoding="UTF-8"?>\n'
        '<parametersSet xmlns="http://www.rte-france.com/dynawo">\n'
    )
    for generator in GENERATORS:
        machine_parameters = {**MACHINE_PARAMETERS, "generator_H": INERTIA_CONSTANTS[generator]}
        for suffix, regulator_parameters in SCENARIO_PARAMETERS.items():
            xml_dynamic_model_parameters += parameter_set(
                f"{generator}{suffix}", {**machine_parameters, **regulator_parameters}, "generator"
            )
    for load in LOADS:
        xml_dynamic_model_parameters += parameter_set(load, LOAD_PARAMETERS, "load")
    xml_dynamic_model_parameters += "</parametersSet>\n"

    xml_static_network_parameters = """<?xml version="1.0" encoding="UTF-8"?>
<parametersSet xmlns="http://www.rte-france.com/dynawo">
    <set id="Network">
        <par type="DOUBLE" name="capacitor_no_reclosing_delay" value="300.0"/>
        <par type="DOUBLE" name="dangling_line_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="line_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="load_Tp" value="90.0"/>
        <par type="DOUBLE" name="load_Tq" value="90.0"/>
        <par type="DOUBLE" name="load_alpha" value="1.0"/>
        <par type="DOUBLE" name="load_alphaLong" value="0.0"/>
        <par type="DOUBLE" name="load_beta" value="2.0"/>
        <par type="DOUBLE" name="load_betaLong" value="0.0"/>
        <par type="BOOL" name="load_isControllable" value="false"/>
        <par type="BOOL" name="load_isRestorative" value="false"/>
        <par type="DOUBLE" name="load_zPMax" value="100.0"/>
        <par type="DOUBLE" name="load_zQMax" value="100.0"/>
        <par type="DOUBLE" name="reactance_no_reclosing_delay" value="0.0"/>
        <par type="DOUBLE" name="transformer_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="transformer_t1st_HT" value="30.0"/>
        <par type="DOUBLE" name="transformer_t1st_THT" value="30.0"/>
        <par type="DOUBLE" name="transformer_tNext_HT" value="10.0"/>
        <par type="DOUBLE" name="transformer_tNext_THT" value="10.0"/>
        <par type="DOUBLE" name="transformer_tolV" value="0.0149999997"/>
        <par type="BOOL" name="BUS_8_hasShortCircuitCapabilities" value="true"/>
    </set>
</parametersSet>
"""

    export_data_dir = "getting_started_data"
    os.makedirs(export_data_dir, exist_ok=True)

    with open(os.path.join(export_data_dir, "Base_Case.par"), "w") as file:
        file.write(xml_dynamic_model_parameters)

    with open(os.path.join(export_data_dir, "Network.par"), "w") as file:
        file.write(xml_static_network_parameters)

    print("Physical and network parameters successfully saved.")


# Execute file generation
generate_parameter_files()

## Step 7: Execute the Dynamic Simulations

Introduce the event of the Getting Started IV notebook: a three-phase short-circuit at `BUS_8`, in the middle of the tie, at $t = 1$ s, cleared after 100 ms when the circuit `LINE_8_9_1` is tripped. It is simulated:

1. with the standard models precompiled in the Dynawo model database, without and with PSS (as in the Getting Started IV notebook);
2. with the standard models compiled by this notebook: both give the same curves;
3. with the variant compiled by this notebook: the speed governor stops the drift of the frequency that follows the event with a constant mechanical power.

PyPowSyBl writes the final state of a simulation into the network, so each simulation starts from a copy of the Loadflow solution.

In [ ]:
# --- 13. EXECUTE THE DYNAMIC SIMULATIONS AND PLOT ---
STANDARD_SCENARIOS = ["Without PSS", "With PSS"]


def run_simulation(mapping, model_database=None, additional_models_file=None):
    """Runs the fault scenario with a dynamic mapping and returns the curves."""
    # Map events (fault at BUS_8, in the middle of the tie, cleared by tripping a circuit)
    event_mapping = dyn.EventMapping()
    event_mapping.add_node_fault(
        static_id="BUS_8", start_time=1.0, fault_time=0.1, r_pu=0.0, x_pu=0.0001
    )
    event_mapping.add_disconnection(static_id="LINE_8_9_1", start_time=1.1)

    # Map output curves: rotor speed and active power of the generators, voltages of the tie
    output_mapping = dyn.OutputVariableMapping()
    for generator in GENERATORS:
        output_mapping.add_dynamic_model_curves(
            generator, ["generator_omegaPu_value", "generator_PGenPu"]
        )
    for bus in ("BUS_7", "BUS_8", "BUS_9"):
        output_mapping.add_standard_model_curves(bus, "U_value")

    # The other simulation parameters come from the PyPowSyBl configuration (Step 0)
    provider_parameters = {}
    if additional_models_file is not None:
        provider_parameters["additionalModelsFile"] = str(additional_models_file)
    sim_parameters = dyn.Parameters(
        start_time=0.0, stop_time=20.0, provider_parameters=provider_parameters
    )

    # Every simulation starts from a copy of the Loadflow solution
    network = pp.network.load_from_binary_buffer(
        power_network.save_to_binary_buffer(format="XIIDM")
    )
    with dynawo_model_database(model_database):
        sim_results = dyn.Simulation().run(
            network,
            model_mapping=mapping,
            event_mapping=event_mapping,
            timeseries_mapping=output_mapping,
            parameters=sim_parameters,
        )
    if sim_results.status().name != "SUCCESS":
        raise RuntimeError(f"Simulation failed. Status: {sim_results.status_text()}")
    return to_seconds(sim_results.curves())


def to_seconds(curves):
    """Indexes the curves by the time in s (PyPowSyBl gives timestamps from t = 0)."""
    curves.index = (curves.index - curves.index[0]).total_seconds()
    return curves


def speed_difference(curves):
    """Rotor speed difference between GEN_1 (area 1) and GEN_3 (area 2)."""
    return curves["GEN_1_generator_omegaPu_value"] - curves["GEN_3_generator_omegaPu_value"]


def mean_rotor_speed(curves):
    """Mean rotor speed of the generators."""
    return curves[[f"{generator}_generator_omegaPu_value" for generator in GENERATORS]].mean(axis=1)


def plot_simulation_results(precompiled_curves, compiled_curves, variant_curves):
    """Plots the inter-area oscillation and the mean rotor speed of the simulations."""
    figure, axes = plt.subplots(1, 3, figsize=(18, 5))
    for scenario in STANDARD_SCENARIOS:
        axes[0].plot(
            precompiled_curves[scenario].index,
            speed_difference(precompiled_curves[scenario]),
            label=f"{scenario} (precompiled)",
        )
        axes[0].plot(
            compiled_curves[scenario].index,
            speed_difference(compiled_curves[scenario]),
            linestyle="--",
            label=f"{scenario} (compiled)",
        )
    for curves, style, label in (
        (compiled_curves["With PSS"], "-", "With PSS: constant mechanical power"),
        (variant_curves, "--", "With PSS and governor (GoverProportional)"),
    ):
        axes[1].plot(curves.index, speed_difference(curves), linestyle=style, label=label)
        axes[2].plot(curves.index, mean_rotor_speed(curves), linestyle=style, label=label)

    axes[0].set_title("Standard models: rotor speed of GEN_1 minus GEN_3")
    axes[1].set_title("Compiled models: rotor speed of GEN_1 minus GEN_3")
    axes[2].set_title("Compiled models: mean rotor speed of the generators")
    axes[0].set_ylabel("Speed difference (pu)")
    axes[1].set_ylabel("Speed difference (pu)")
    axes[2].set_ylabel("Rotor speed (pu)")
    for axis in axes:
        axis.set_xlabel("Time (Seconds)")
        axis.grid(True)
        axis.legend(fontsize=8)
    figure.suptitle("Transient Response: Kundur Two-Area System")
    figure.tight_layout()
    plt.show()


precompiled_curves, compiled_curves = {}, {}
for scenario in STANDARD_SCENARIOS:
    print(f"{scenario}: standard models precompiled in the Dynawo model database...")
    precompiled_curves[scenario] = run_simulation(scenario_mappings[scenario])
    print(f"{scenario}: standard models compiled by this notebook...")
    compiled_curves[scenario] = run_simulation(scenario_mappings[scenario], MODEL_DATABASE)
print("With PSS and governor: variant compiled by this notebook (GoverProportional)...")
variant_curves = run_simulation(
    scenario_mappings["With PSS and governor"], MODEL_DATABASE, ADDITIONAL_MODELS_FILE
)

print()
for scenario in STANDARD_SCENARIOS:
    difference = abs(precompiled_curves[scenario].values - compiled_curves[scenario].values).max()
    print(
        f"Maximum difference between the precompiled and the compiled models ({scenario}): {difference:.2e}"
    )
plot_simulation_results(precompiled_curves, compiled_curves, variant_curves)

## Step 8: Analyse the Inter-Area Oscillation

After the fault, the generators of area 1 swing against those of area 2: the inter-area mode, visible in the difference between the rotor speeds of `GEN_1` and `GEN_3`. Its frequency is estimated from the duration of its cycles (between consecutive upward zero crossings), and its damping ratio $\zeta$ from the logarithmic decrement $\delta$ between the peaks of the first and the last of its $n + 1$ cycles (at least two cycles are needed):

$$\delta = \frac{1}{n} \ln \frac{x_0}{x_n}, \qquad \zeta = \frac{\delta}{\sqrt{4\pi^2 + \delta^2}}$$

The final mean rotor speed of the generators shows the frequency reached after the event. The governor of the variant, which has no time constant, also damps the speed oscillations of the machines: when they fade out within less than two cycles, the mode cannot be estimated (NaN).

In [ ]:
# --- 14. ANALYSE THE INTER-AREA OSCILLATION ---
def analyse_inter_area_mode(curves, start_time=2.0):
    """
    Estimates the frequency and the damping ratio of the inter-area mode from the rotor speed
    difference between the areas, once the fault is cleared, and reads the final mean speed.
    """
    oscillation = speed_difference(curves)
    oscillation = oscillation[oscillation.index >= start_time]
    values = (oscillation - oscillation.mean()).values
    times = oscillation.index.values

    # Upward zero crossings delimit the cycles of the oscillation
    crossings = np.where((values[:-1] < 0) & (values[1:] >= 0))[0]
    peaks = [values[first:last].max() for first, last in pairwise(crossings)]

    # The mode can only be estimated from two cycles or more
    frequency = damping_ratio = np.nan
    if len(peaks) >= 2:
        frequency = (len(crossings) - 1) / (times[crossings[-1]] - times[crossings[0]])
        decrement = np.log(peaks[0] / peaks[-1]) / (len(peaks) - 1)
        damping_ratio = 100 * decrement / np.sqrt(4 * np.pi**2 + decrement**2)

    return {
        "Frequency (Hz)": round(frequency, 3),
        "Damping ratio (%)": round(damping_ratio, 2),
        "Cycles": len(peaks),
        "First peak (pu)": peaks[0] if peaks else np.nan,
        "Last peak (pu)": peaks[-1] if peaks else np.nan,
        "Final mean speed (pu)": round(mean_rotor_speed(curves).iloc[-1], 5),
    }


compiled_scenario_curves = {**compiled_curves, "With PSS and governor": variant_curves}
mode_analysis = pd.DataFrame.from_dict(
    {
        scenario: analyse_inter_area_mode(curves)
        for scenario, curves in compiled_scenario_curves.items()
    },
    orient="index",
)
display(mode_analysis)

### Observations

- **Compilation:** the four models compile from their architecture files, and the compiled standard models give exactly the same curves as the precompiled ones (maximum difference of 0), without and with PSS.
- **Governor:** with the proportional governor of the variant, the mean rotor speed settles at about 1.0006 pu after the event, instead of rising steadily with a constant mechanical power. The proportional control leaves a small permanent frequency deviation (about 0.03 Hz at 50 Hz).
- **Damping:** the governor, which has no time constant, also damps the inter-area oscillation: it fades out within about one cycle, too few cycles to estimate the mode.